# OPP-115 → جدول تدريب الـ Clause Classifier

يبني من `OPP-115/` جدول **فقرة واحدة في كل صف** (نص + فئات الخصوصية العشر) مقسّم **حسب السياسة** (train / validation / test).

**المدخلات:** `OPP-115/annotations/` و`OPP-115/sanitized_policies/` و`OPP-115/documentation/`  
**المخرجات** في `opp115_output/`:

| الملف | الفائدة |
|---|---|
| `opp115_segments.csv` | الجدول الرئيسي: نص الفقرة + 10 أعمدة label + القسم (split) |
| `opp115_segment_votes.csv` | عدد المعلّقين (0–3) الذين اختاروا كل فئة، لتغيير قاعدة الأغلبية بدون إعادة الاستخراج |
| `opp115_policy_splits.csv` | كل سياسة وأي قسم دخلت |
| `opp115_split_summary.csv` | توزيع الفئات داخل كل قسم |
| `opp115_build_report.json` | إعدادات التشغيل وأرقام الفحوص |

**ملاحظات تحققت منها على البيانات:**
1. رقم السياسة = البادئة الرقمية في اسم الملف (تطابق `policies_opp115.csv`). عمود `policy ID` داخل `annotations` رقم داخلي مختلف ولا يُستخدم.
2. أرقام الـ segment في `annotations` تطابق تقسيم `|||` في `sanitized_policies` لكل الـ115 سياسة.
3. كل سياسة عليها 3 معلّقين بالضبط.

In [ ]:
from pathlib import Path
import csv, re, json, html
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

csv.field_size_limit(10**9)

# GitHub-portable root: run from repository root OR its notebooks/ folder.
_WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = _WORKDIR.parent if _WORKDIR.name.lower() == "notebooks" else _WORKDIR

OPP_DIR    = PROJECT_ROOT / "OPP-115"          # المجلد الناتج من فك OPP-115_clean.zip (أو مسار كامل: Path(r"C:\...\OPP-115"))
OUTPUT_DIR = PROJECT_ROOT / "opp115_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- قرارات الفريق (غيّرها هنا فقط) ----
MIN_VOTES = 2          # فئة تُحسب للفقرة إذا اختارها على الأقل هذا العدد من المعلّقين الثلاثة
SEED      = 42
N_VAL, N_TEST = 11, 12 # عدد السياسات (من 115): train = 92
N_TRIALS  = 5000       # عدد المحاولات العشوائية لاختيار تقسيم متوازن
MIN_EVAL_SEGMENTS_PER_CATEGORY = 3   # كل فئة لازم تظهر على الأقل بهذا العدد في validation و test

print("OPP_DIR exists:", OPP_DIR.exists(), OPP_DIR)
print("Output directory:", OUTPUT_DIR.resolve())

## 1) الفئات والسياسات

In [ ]:
def read_csv(path):
    with open(path, encoding="utf-8", errors="replace", newline="") as f:
        return list(csv.reader(f))

assert OPP_DIR.exists(), f"OPP_DIR not found: {OPP_DIR}"
DOCS = OPP_DIR / "documentation"
xml = (DOCS / "categories-july30.xml").read_text(encoding="utf-8", errors="replace")
CATEGORIES = [m.group(1).strip() for m in re.finditer(r"<category>\s*<name>([^<]+)</name>", xml)]
assert len(CATEGORIES) == 10, CATEGORIES

SHORT = {   # أسماء أعمدة قصيرة للـ labels
    "First Party Collection/Use": "first_party_collection_use",
    "Third Party Sharing/Collection": "third_party_sharing_collection",
    "User Choice/Control": "user_choice_control",
    "User Access, Edit and Deletion": "user_access_edit_deletion",
    "Data Retention": "data_retention",
    "Data Security": "data_security",
    "Policy Change": "policy_change",
    "Do Not Track": "do_not_track",
    "International and Specific Audiences": "international_specific_audiences",
    "Other": "other",
}
assert set(SHORT) == set(CATEGORIES)
LABEL_COLS = [SHORT[c] for c in CATEGORIES]

policy_url = {r[0]: r[1] for r in read_csv(DOCS / "policies_opp115.csv")[1:] if r}
site_name  = {r[3]: r[2] for r in read_csv(DOCS / "websites_opp115.csv")[1:] if len(r) > 3}

ann_files = sorted((OPP_DIR / "annotations").glob("*.csv"))
print("annotation files found:", len(ann_files), "| policies in policies_opp115.csv:", len(policy_url))
assert len(ann_files) == 115, f"expected 115 files in {OPP_DIR / 'annotations'}, found {len(ann_files)}"

uid_of = {f.stem: f.stem.split("_")[0] for f in ann_files}       # البادئة الرقمية = Policy UID
diff = set(uid_of.values()) ^ set(policy_url)
assert not diff, f"policy IDs do not match policies_opp115.csv: {sorted(diff)[:10]}"
print(len(CATEGORIES), "categories |", len(ann_files), "policies")

## 2) استخراج الفقرات من `sanitized_policies` وتنظيفها
الفاصل بين الفقرات هو `|||`. التنظيف: حذف وسوم HTML (`<br>` و`<li>` …)، تحويل رموز HTML، توحيد المسافات. **لا نغيّر الكلمات.**

In [ ]:
TAG = re.compile(r"<[^>]+>")
def clean_text(t):
    return re.sub(r"\s+", " ", html.unescape(TAG.sub(" ", t))).strip()

segments = {}     # (policy_uid, segment_id) -> text
for f in ann_files:
    uid = uid_of[f.stem]
    raw = (OPP_DIR / "sanitized_policies" / f"{f.stem}.html").read_text(encoding="utf-8", errors="replace")
    for i, part in enumerate(raw.split("|||")):
        segments[(uid, i)] = clean_text(part)

empty = [k for k, v in segments.items() if not v]
print("segments:", len(segments), "| empty after cleaning:", len(empty))

## 3) أصوات المعلّقين → labels
لكل (سياسة، فقرة، فئة) نحسب عدد المعلّقين المختلفين الذين اختاروها (0 إلى 3). الفئة تدخل label الفقرة إذا الأصوات ≥ `MIN_VOTES`. الفقرة التي ما حصلت فيها أي فئة على هذا العدد تُعلَّم `no_majority` وتبقى في الملف.

In [ ]:
votes = defaultdict(lambda: defaultdict(set))
annotators = defaultdict(set)
for f in ann_files:
    uid = uid_of[f.stem]
    for r in read_csv(f):
        seg_id, cat, annotator = int(r[4]), r[5], r[2]
        assert cat in SHORT, cat
        assert (uid, seg_id) in segments, f"segment {seg_id} not found in {f.stem}"   # فحص التطابق
        votes[(uid, seg_id)][cat].add(annotator)
        annotators[uid].add(annotator)
assert all(len(a) == 3 for a in annotators.values()), "each policy must have exactly 3 annotators"

rows, vote_rows = [], []
for (uid, seg_id), text in sorted(segments.items(), key=lambda kv: (int(kv[0][0]), kv[0][1])):
    v = {c: len(votes[(uid, seg_id)].get(c, ())) for c in CATEGORIES}
    chosen = [c for c in CATEGORIES if v[c] >= MIN_VOTES]
    rows.append({
        "policy_uid": int(uid),
        "site_name": site_name.get(uid, ""), "segment_id": seg_id, "text": text, "n_chars": len(text),
        "categories": " | ".join(chosen), "n_labels": len(chosen),
        "label_status": "majority" if chosen else "no_majority",
        **{SHORT[c]: int(c in chosen) for c in CATEGORIES},
    })
    vote_rows.append({"policy_uid": int(uid), "segment_id": seg_id, **{"votes_" + SHORT[c]: v[c] for c in CATEGORIES}})

file_of = {u: s for s, u in uid_of.items()}
df = pd.DataFrame(rows)
df["policy_file"] = df["policy_uid"].astype(str).map(file_of)
df_votes = pd.DataFrame(vote_rows)
print("segments:", len(df), "| no_majority:", int((df.label_status == "no_majority").sum()))
print("labels per segment:", df.n_labels.value_counts().sort_index().to_dict())

## 4) التقسيم حسب السياسة (train / validation / test)
الوحدة هي **السياسة كاملة**، فلا تنقسم سياسة بين قسمين. نجرّب `N_TRIALS` تقسيمًا عشوائيًا (بذرة ثابتة) ونختار الأقرب لنسبة 80/10/10 من ناحية عدد الفقرات **وكل فئة**، بشرط أن تظهر كل فئة (حتى `Do Not Track`) في validation وtest بحد أدنى.

In [ ]:
uids = sorted(df.policy_uid.unique())
labeled = df[df.label_status == "majority"]
M = np.array([labeled[labeled.policy_uid == u][LABEL_COLS].sum().values for u in uids])   # policies x categories
S = np.array([(df.policy_uid == u).sum() for u in uids])                                  # segments per policy
total_cat, total_seg = M.sum(0), S.sum()

rng = np.random.default_rng(SEED)
best = None
for _ in range(N_TRIALS):
    perm = rng.permutation(len(uids))
    val, test = perm[:N_VAL], perm[N_VAL:N_VAL + N_TEST]
    mv, mt = M[val].sum(0), M[test].sum(0)
    if (mv < MIN_EVAL_SEGMENTS_PER_CATEGORY).any() or (mt < MIN_EVAL_SEGMENTS_PER_CATEGORY).any():
        continue
    cost = (np.abs(mv / total_cat - 0.10).sum() + np.abs(mt / total_cat - 0.10).sum()
            + 3 * (abs(S[val].sum() / total_seg - 0.10) + abs(S[test].sum() / total_seg - 0.10)))
    if best is None or cost < best[0]:
        best = (cost, val, test)
assert best is not None, "no valid split found; lower MIN_EVAL_SEGMENTS_PER_CATEGORY"

split_of = {u: "train" for u in uids}
for i in best[1]: split_of[uids[i]] = "validation"
for i in best[2]: split_of[uids[i]] = "test"
df["split"] = df.policy_uid.map(split_of)

# فحص التسرب: كل سياسة في قسم واحد فقط
assert df.groupby("policy_uid").split.nunique().max() == 1
print(df.groupby("split").agg(policies=("policy_uid", "nunique"), segments=("segment_id", "size")))

In [ ]:
# توزيع الفئات في كل قسم (الفقرات ذات label أغلبية)
summary = (df[df.label_status == "majority"].groupby("split")[LABEL_COLS].sum().T
           .reindex(columns=["train", "validation", "test"]))
summary["total"] = summary.sum(axis=1)
for s in ["train", "validation", "test"]:
    summary[s + "_%"] = (summary[s] / summary["total"] * 100).round(1)
summary.index.name = "category"
display(summary)

## 5) حفظ المخرجات والفحوص النهائية

In [ ]:
seg_cols = ["policy_uid", "policy_file", "site_name", "segment_id", "split", "text", "n_chars",
            "categories", "n_labels", "label_status"] + LABEL_COLS
df[seg_cols].to_csv(OUTPUT_DIR / "opp115_segments.csv", index=False, encoding="utf-8-sig")
df_votes.to_csv(OUTPUT_DIR / "opp115_segment_votes.csv", index=False, encoding="utf-8-sig")

pol = (df.groupby("policy_uid").agg(policy_file=("policy_file", "first"), site_name=("site_name", "first"),
                                    n_segments=("segment_id", "size"), split=("split", "first")).reset_index())
pol["policy_url"] = pol.policy_uid.astype(str).map(policy_url)
pol.to_csv(OUTPUT_DIR / "opp115_policy_splits.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUTPUT_DIR / "opp115_split_summary.csv", encoding="utf-8-sig")

report = {
    "policies": int(df.policy_uid.nunique()), "segments": int(len(df)),
    "segments_no_majority": int((df.label_status == "no_majority").sum()),
    "min_votes": MIN_VOTES, "seed": SEED, "n_trials": N_TRIALS,
    "policies_per_split": pol.split.value_counts().to_dict(),
    "segments_per_split": df.split.value_counts().to_dict(),
    "multi_label_share_of_labeled": round(float((labeled.n_labels > 1).mean()), 3),
}
(OUTPUT_DIR / "opp115_build_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

assert report["policies"] == 115 and report["segments"] == 3792
assert sum(report["policies_per_split"].values()) == 115
assert df.text.str.len().min() > 0
print(json.dumps(report, indent=2))
print("Saved to:", OUTPUT_DIR.resolve())